In [166]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [167]:
df = pd.read_csv("data/HR/train.csv")
df.head()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won?,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73,0


In [168]:
df.duplicated().sum()   

np.int64(0)

In [169]:
df.isnull().mean().mean() * 100

np.float64(0.9169071331529367)

In [170]:
df['previous_year_rating'] = df['previous_year_rating'].fillna(0)

In [171]:
df['education'] = df['education'].fillna('missing')

In [172]:
df.isnull().mean().mean() * 100

np.float64(0.0)

In [173]:
df.columns

Index(['employee_id', 'department', 'region', 'education', 'gender',
       'recruitment_channel', 'no_of_trainings', 'age', 'previous_year_rating',
       'length_of_service', 'awards_won?', 'avg_training_score',
       'is_promoted'],
      dtype='str')

In [174]:
df['education'].unique()

<StringArray>
['Master's & above', 'Bachelor's', 'missing', 'Below Secondary']
Length: 4, dtype: str

In [175]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder , StandardScaler, OrdinalEncoder, PowerTransformer

#Categorical columns department, region, education, gender, recruitment_channel need encoding. region by Target Encoding,
#  Department/gender by OHE. Numerical columns age, length_of_service, avg_training_score, no_of_trainings already good range, can standardize later.

processor = ColumnTransformer(
    transformers=[
        ('pw', PowerTransformer(), ['length_of_service']),
        ('num', StandardScaler(), ['age', 'avg_training_score']),
        ('target', TargetEncoder(), ['region']),
        ('cat', OneHotEncoder(), ['department', 'gender', 'recruitment_channel']),
        ('ordinal', OrdinalEncoder(categories=[[ 'missing','Below Secondary', 'Bachelor\'s', 'Master\'s & above']]), ['education'])
    ],
    remainder='passthrough'
)

In [176]:
X = df.drop('is_promoted', axis=1)
y = df['is_promoted']

In [177]:
processor.fit(X, y)
X_processed = processor.transform(X)

In [178]:
df = pd.DataFrame(X_processed,index=X.index,columns=processor.get_feature_names_out())
df.head()

,pw__length_of_service,num__age,num__avg_training_score,target__region,cat__department_Analytics,cat__department_Finance,cat__department_HR,cat__department_Legal,cat__department_Operations,cat__department_Procurement,...,cat__gender_f,cat__gender_m,cat__recruitment_channel_other,cat__recruitment_channel_referred,cat__recruitment_channel_sourcing,ordinal__education,remainder__employee_id,remainder__no_of_trainings,remainder__previous_year_rating,remainder__awards_won?
0,0.762071,0.025598,-1.075931,0.106540,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,3.0,65438.0,1.0,5.0,0.0
1,-0.246487,-0.627135,-0.253282,0.114182,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,2.0,65141.0,1.0,5.0,0.0
2,0.561785,-0.104948,-1.001145,0.060661,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,2.0,7513.0,1.0,3.0,0.0
3,1.101233,0.547785,-1.001145,0.116560,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,2.0,2542.0,2.0,1.0,0.0
4,-1.141600,1.331064,0.718939,0.063282,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,2.0,48945.0,1.0,3.0,0.0


In [179]:
df.columns

Index(['pw__length_of_service', 'num__age', 'num__avg_training_score',
       'target__region', 'cat__department_Analytics',
       'cat__department_Finance', 'cat__department_HR',
       'cat__department_Legal', 'cat__department_Operations',
       'cat__department_Procurement', 'cat__department_R&D',
       'cat__department_Sales & Marketing', 'cat__department_Technology',
       'cat__gender_f', 'cat__gender_m', 'cat__recruitment_channel_other',
       'cat__recruitment_channel_referred',
       'cat__recruitment_channel_sourcing', 'ordinal__education',
       'remainder__employee_id', 'remainder__no_of_trainings',
       'remainder__previous_year_rating', 'remainder__awards_won?'],
      dtype='str')

In [180]:
df.columns = df.columns.str.replace('num__','')
df.columns = df.columns.str.replace('cat__','')
df.columns = df.columns.str.replace('ordinal__','')
df.columns = df.columns.str.replace('remainder__','')
df.columns  = df.columns.str.replace('target__','')
df.columns = df.columns.str.replace('pw__','')

In [181]:
df.columns.__len__()

23

In [182]:
df['is_promoted'] = y

In [183]:
df.shape

(54808, 24)

In [184]:
df.head()

,length_of_service,age,avg_training_score,region,department_Analytics,department_Finance,department_HR,department_Legal,department_Operations,department_Procurement,...,gender_m,recruitment_channel_other,recruitment_channel_referred,recruitment_channel_sourcing,education,employee_id,no_of_trainings,previous_year_rating,awards_won?,is_promoted
0,0.762071,0.025598,-1.075931,0.106540,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,3.0,65438.0,1.0,5.0,0.0,0
1,-0.246487,-0.627135,-0.253282,0.114182,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,2.0,65141.0,1.0,5.0,0.0,0
2,0.561785,-0.104948,-1.001145,0.060661,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,2.0,7513.0,1.0,3.0,0.0,0
3,1.101233,0.547785,-1.001145,0.116560,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,2.0,2542.0,2.0,1.0,0.0,0
4,-1.141600,1.331064,0.718939,0.063282,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,2.0,48945.0,1.0,3.0,0.0,0


In [185]:
import numpy as np

def create_numeric_features(df):
    df = df.copy()

    # 1. Experience ratio
    df["experience_ratio"] = df["length_of_service"] / df["age"]

    # 2. Joining age
    df["joining_age"] = df["age"] - df["length_of_service"]

    # 3. Training efficiency
    df["training_efficiency"] = df["avg_training_score"] / df["no_of_trainings"]
    df["training_efficiency"] = df["training_efficiency"].replace([np.inf], 0)

    # 4. Training frequency
    df["training_frequency"] = df["no_of_trainings"] / (df["length_of_service"] + 1)

    # 5. Performance score
    df["performance_score"] = df["previous_year_rating"] * df["avg_training_score"]

    # 6. Award score
    df["award_score"] = df["awards_won?"] * df["avg_training_score"]

    # 7. Score per age
    df["score_per_age"] = df["avg_training_score"] / df["age"]

    # 8. Service score interaction
    df["service_score"] = df["length_of_service"] * df["avg_training_score"]

    # 9. Age-service interaction
    df["age_service_interaction"] = df["age"] * df["length_of_service"]

    # 10. High performer flag (binary numeric)
    df["high_performer"] = (
        (df["previous_year_rating"] >= 4) &
        (df["avg_training_score"] > 80)
    ).astype(int)

    return df

In [186]:
df  = create_numeric_features(df)

In [187]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('is_promoted', axis=1), df['is_promoted'], test_size=0.2, random_state=42)

In [188]:
from sklearn.linear_model import LogisticRegression
from  sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      1.00      0.96     10054
           1       0.78      0.15      0.25       908

    accuracy                           0.93     10962
   macro avg       0.86      0.57      0.61     10962
weighted avg       0.92      0.93      0.90     10962

[[10016    38]
 [  770   138]]
Accuracy: 0.9262908228425469


/home/tarun.nagpal@simform.dom/Documents/EDA/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [189]:
X = df.drop('is_promoted', axis=1)
y = df['is_promoted']

In [190]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X, y)

In [191]:
X_resampled.head()

,length_of_service,age,avg_training_score,region,department_Analytics,department_Finance,department_HR,department_Legal,department_Operations,department_Procurement,...,experience_ratio,joining_age,training_efficiency,training_frequency,performance_score,award_score,score_per_age,service_score,age_service_interaction,high_performer
0,0.762071,0.025598,-1.075931,0.106540,0.0,0.0,0.0,0.0,0.0,0.0,...,29.770532,-0.736473,-1.075931,0.567514,-5.379657,-0.0,-42.031579,-0.819936,0.019508,0
1,-0.246487,-0.627135,-0.253282,0.114182,0.0,0.0,0.0,0.0,1.0,0.0,...,0.393036,-0.380648,-0.253282,1.327117,-1.266412,-0.0,0.403872,0.062431,0.154580,0
2,0.561785,-0.104948,-1.001145,0.060661,0.0,0.0,0.0,0.0,0.0,0.0,...,-5.352963,-0.666734,-1.001145,0.640293,-3.003436,-0.0,9.539400,-0.562428,-0.058958,0
3,1.101233,0.547785,-1.001145,0.116560,0.0,0.0,0.0,0.0,0.0,0.0,...,2.010339,-0.553448,-0.500573,0.951822,-1.001145,-0.0,-1.827626,-1.102494,0.603238,0
4,-1.141600,1.331064,0.718939,0.063282,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.857660,2.472665,0.718939,-7.062135,2.156817,0.0,0.540124,-0.820741,-1.519543,0


In [192]:
df = X_resampled.copy()
df['is_promoted'] = y_resampled

In [193]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('is_promoted', axis=1), df['is_promoted'], test_size=0.2, random_state=42)

In [194]:
from sklearn.linear_model import LogisticRegression
from  sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.70      0.70     10197
           1       0.69      0.69      0.69      9859

    accuracy                           0.70     20056
   macro avg       0.70      0.70      0.70     20056
weighted avg       0.70      0.70      0.70     20056

[[7178 3019]
 [3010 6849]]
Accuracy: 0.6993917032309533


/home/tarun.nagpal@simform.dom/Documents/EDA/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [195]:
X = df.drop('is_promoted', axis=1)
y = df['is_promoted']

In [198]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)

X_new = selector.fit_transform(X)

selected_features = X.columns[selector.get_support()]

len(selected_features)

31

In [200]:
X = X[selected_features]
X.shape

(100280, 31)

In [203]:
from sklearn.feature_selection import RFE

r = RFE(estimator=LogisticRegression(), n_features_to_select=20)
r.fit(X, y)
X_new = r.transform(X)
r.get_support()

/home/tarun.nagpal@simform.dom/Documents/EDA/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/tarun.nagpal@simform.dom/Documents/EDA/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
   

array([ True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,  True,  True, False, False, False, False, False,
        True, False, False, False])

In [207]:
selected_features = X.columns[r.get_support()]
(selected_features)

Index(['length_of_service', 'avg_training_score', 'department_Analytics',
       'department_Finance', 'department_HR', 'department_Legal',
       'department_Operations', 'department_Procurement', 'department_R&D',
       'department_Sales & Marketing', 'department_Technology', 'gender_f',
       'gender_m', 'recruitment_channel_other', 'recruitment_channel_referred',
       'recruitment_channel_sourcing', 'education', 'previous_year_rating',
       'awards_won?', 'award_score'],
      dtype='str')

In [216]:
from sklearn.linear_model import Lasso

l = Lasso(alpha=0.01)

l.fit(X, y)


print(len(l.feature_names_in_))
count = 0
for feature, coef in zip(l.feature_names_in_, l.coef_):
    count = count + 1 if coef != 0 else count
    print(feature, coef)
print("Number of features selected:", count)

31
length_of_service -0.028149560928231607
age -0.007868648653318471
avg_training_score 0.12849136133164746
department_Analytics -0.06705034951514405
department_Finance 0.0
department_HR 0.0
department_Legal -0.0
department_Operations 0.027278550180050393
department_Procurement -0.0
department_R&D -0.0
department_Sales & Marketing 0.12380257970325512
department_Technology -0.0
gender_f 0.0
gender_m -0.0
recruitment_channel_other -0.0
recruitment_channel_referred 0.0
recruitment_channel_sourcing -0.0
education 0.00941971836595635
employee_id -1.0302582927709914e-07
no_of_trainings -0.012797765307552488
previous_year_rating 0.07512363856339482
awards_won? 0.17387079577171466
experience_ratio 0.000421504069868163
joining_age 0.0
training_efficiency 0.0
training_frequency 0.00045027546003596997
performance_score 0.012725538883978685
award_score 0.0
score_per_age -0.0006205524748041032
service_score -0.0
age_service_interaction 0.008403288613792557
Number of features selected: 16


In [212]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest

selector = SelectKBest(score_func=mutual_info_classif, k=15)

selector.fit(X, y)

selected_features = X.columns[selector.get_support()]

(selected_features)

Index(['length_of_service', 'age', 'avg_training_score',
       'recruitment_channel_other', 'recruitment_channel_sourcing',
       'education', 'previous_year_rating', 'experience_ratio', 'joining_age',
       'training_efficiency', 'training_frequency', 'performance_score',
       'score_per_age', 'service_score', 'age_service_interaction'],
      dtype='str')